# IMPORT LIBRARIES

In [2]:
import pandas as pd
import numpy as np
from mlxtend.frequent_patterns import fpgrowth , association_rules , apriori
from mlxtend.preprocessing import TransactionEncoder

/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [3]:

df = pd.read_excel('/Users/sahil_jangid/codes/machine learning/MBA(market_basket_analysis)/Online Retail.xlsx')

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 40.0+ MB


In [5]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


# CLEANING

In [6]:
df = df[['InvoiceNo' ,'Description' , 'Quantity' ]]

In [7]:
df.head()

,InvoiceNo,Description,Quantity
0,536365,WHITE HANGING HEART T-LIGHT HOLDER,6
1,536365,WHITE METAL LANTERN,6
2,536365,CREAM CUPID HEARTS COAT HANGER,8
3,536365,KNITTED UNION FLAG HOT WATER BOTTLE,6
4,536365,RED WOOLLY HOTTIE WHITE HEART.,6


In [8]:
df = df.dropna(subset=['Description'])

In [9]:
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]

In [10]:
df.shape

(531167, 3)

In [11]:
df = df[df['Quantity'] > 0]

In [12]:
df = df.drop(columns=['Quantity'])

In [13]:
df.head()

,InvoiceNo,Description
0,536365,WHITE HANGING HEART T-LIGHT HOLDER
1,536365,WHITE METAL LANTERN
2,536365,CREAM CUPID HEARTS COAT HANGER
3,536365,KNITTED UNION FLAG HOT WATER BOTTLE
4,536365,RED WOOLLY HOTTIE WHITE HEART.


# TRANSFORMING THE DATA

In [14]:
transactions = df.groupby('InvoiceNo')['Description'].apply(list).tolist()

In [15]:
transactions[0]

['WHITE HANGING HEART T-LIGHT HOLDER',
 'WHITE METAL LANTERN',
 'CREAM CUPID HEARTS COAT HANGER',
 'KNITTED UNION FLAG HOT WATER BOTTLE',
 'RED WOOLLY HOTTIE WHITE HEART.',
 'SET 7 BABUSHKA NESTING BOXES',
 'GLASS STAR FROSTED T-LIGHT HOLDER']

# ONE HOT ENCODING

In [16]:
te = TransactionEncoder()
te_data = te.fit_transform(transactions)

In [17]:
basket = pd.DataFrame(te_data, columns= te.columns_)

In [18]:
basket.head()

,4 PURPLE FLOCK DINNER CANDLES,50'S CHRISTMAS GIFT BAG LARGE,DOLLY GIRL BEAKER,I LOVE LONDON MINI BACKPACK,I LOVE LONDON MINI RUCKSACK,NINE DRAWER OFFICE TIDY,OVAL WALL MIRROR DIAMANTE,RED SPOT GIFT BAG LARGE,SET 2 TEA TOWELS I LOVE LONDON,SPACEBOY BABY GIFT SET,...,returned,taig adjust,test,to push order througha s stock was,website fixed,wrongly coded 20713,wrongly coded 23343,wrongly marked,wrongly marked 23343,wrongly sold (22719) barcode
0,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


# APPLY MBA

In [ ]:
frequest_items = fpgrowth(
    basket,
    min_support=0.02,
    use_colnames=True,
)
#took 1.1sec


In [ ]:
# to compare the speed of fp growth and aproori
frequest_items2 = apriori(
    basket,
    min_support=0.02,
    use_colnames=True,
)
# took 1.8sec

# GENRATING THE RULES

In [21]:
rules = association_rules(
    frequest_items,
    metric='confidence',
    min_threshold=0.3,
)

# FILTRING THE STRONG RULES

In [22]:
rules = rules[
    (rules['lift'] > 1.5) &
    (rules['confidence'] > 0.5) 
]

In [23]:
rules.columns

Index(['antecedents', 'consequents', 'antecedent support',
       'consequent support', 'support', 'confidence', 'lift',
       'representativity', 'leverage', 'conviction', 'zhangs_metric',
       'jaccard', 'certainty', 'kulczynski'],
      dtype='str')

In [24]:
rules[['antecedents', 'consequents', 'antecedent support','consequent support', 'support', 'confidence', 'lift']].head()

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift
2,frozenset({ALARM CLOCK BAKELIKE GREEN}),frozenset({ALARM CLOCK BAKELIKE RED }),0.048669,0.052195,0.031784,0.653061,12.511932
3,frozenset({ALARM CLOCK BAKELIKE RED }),frozenset({ALARM CLOCK BAKELIKE GREEN}),0.052195,0.048669,0.031784,0.608944,12.511932
4,frozenset({ALARM CLOCK BAKELIKE PINK}),frozenset({ALARM CLOCK BAKELIKE RED }),0.038886,0.052195,0.023341,0.600255,11.500231
6,frozenset({ALARM CLOCK BAKELIKE PINK}),frozenset({ALARM CLOCK BAKELIKE GREEN}),0.038886,0.048669,0.020759,0.533844,10.968864
9,frozenset({WOODEN FRAME ANTIQUE WHITE }),frozenset({WOODEN PICTURE FRAME WHITE FINISH}),0.048222,0.054629,0.026768,0.555098,10.161318
